In [2]:
from steer_core.DataManager import DataManager

from steer_materials.CellMaterials.Base import CurrentCollectorMaterial, SeparatorMaterial
from steer_materials.CellMaterials.Electrode import CathodeMaterial, Binder, ConductiveAdditive, AnodeMaterial

from steer_opencell_design.Components.CurrentCollectors import TabWeldedCurrentCollector, WeldTab
from steer_opencell_design.Formulations.ElectrodeFormulations import CathodeFormulation, AnodeFormulation
from steer_opencell_design.Components.Electrodes import Cathode, Anode
from steer_opencell_design.Components.Separators import Separator
from steer_opencell_design.Constructions.Layups import Laminate
from steer_opencell_design.Constructions.ElectrodeAssemblies.JellyRolls import WoundJellyRoll
from steer_opencell_design.AuxillaryComponents.WindingEquipment import RoundMandrel

from steer_opencell_design import __version__

import pandas as pd
from datetime import datetime as dt

In [3]:
db = DataManager()

In [4]:
db.remove_data(table_name='cells', condition="name = 'Cell 4'")

In [5]:
db.get_table_names()

['cells',
 'anode_materials',
 'binder_materials',
 'cathode_materials',
 'conductive_additive_materials',
 'current_collector_materials',
 'insulation_materials',
 'separator_materials']

In [6]:
# Get current collector materials from the database

current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
separator_material = SeparatorMaterial.from_database("Polypropylene")

In [7]:
# Create the cathode

current_collector_tab = WeldTab(
    material=current_collector_material,
    width=10,
    length=100,
    thickness=10
)

cathode_current_collector = TabWeldedCurrentCollector(
    material=current_collector_material,
    weld_tab=current_collector_tab,
    tab_overhang=20,
    skip_coat_width=20,
    length=4000,
    width=120,
    thickness=12,
    weld_tab_positions=[100, 500, 1500, 2500, 3500, 3800],
    tab_weld_side='b'
)

cathode_active_material = CathodeMaterial.from_database("NaNiMn P2-O3 Composite")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=3,
    mass_loading=15,
    insulation_thickness=3
)


In [8]:
# Create the anode

current_collector_tab = WeldTab(
    material=current_collector_material,
    width=10,
    length=100,
    thickness=10
)

anode_current_collector = TabWeldedCurrentCollector(
    material=current_collector_material,
    weld_tab=current_collector_tab,
    tab_overhang=20,
    skip_coat_width=20,
    length=4010,
    width=123,
    thickness=12,
    weld_tab_positions=[100, 3000],
    tab_weld_side='b'
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=8,
    insulation_thickness=3
)

In [9]:
# make the layup

top_separator = Separator(
    material=separator_material,
    width=286,
    length=4900,
    thickness=25
)

bottom_separator = Separator(
    material=separator_material,
    width=286,
    length=4900,
    thickness=25
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_separator
)


In [11]:
# create the jellyroll assembly

mandrel = RoundMandrel(
    diameter=5,
    length=500,
)

my_jellyroll = WoundJellyRoll(
    laminate=my_layup,
    mandrel=mandrel,
    name="Cell 4",
)


ValueError: zero-size array to reduction operation minimum which has no identity

In [ ]:
my_jellyroll.roll_properties

In [ ]:
my_jellyroll.get_spiral_plot(height=1300, width=1300).show()

In [ ]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_jellyroll.name],
    'object': [my_jellyroll.serialize()],
    'form_factor': ['Cylindrical'],
    'internal_construction': ['Wound'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [ ]:
db.get_data('cells')

,name,object,form_factor,internal_construction,date,version,reference
0,HiNa-NaCR32140-MP10,gASVswMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 10:28:17,0.3.12,Na/Na+
1,QNAS-S40160NL,gASVswMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 10:28:19,0.3.12,Na/Na+
2,QNAS-S40160RL,gASVswMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 10:28:21,0.3.12,Na/Na+
3,UniGrid-NCO32140,gASVzcIAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 10:28:22,0.3.12,Na/Na+
4,Vendor D NFPP,gASVF/0AAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-10-21 10:28:24,0.3.12,Na/Na+
5,Vendor E NFPP,gASVF/0AAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-10-21 10:28:26,0.3.12,Na/Na+
6,Vendor G NFM,gASVswMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 10:28:27,0.3.12,Na/Na+
7,Vendor G NFPP,gASVaPQAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 10:28:29,0.3.12,Na/Na+
8,CBAK-32140NS,gASVswMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 15:18:33,0.3.12,Na/Na+
9,Cell 2,gASVuAkBAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 15:18:34,0.3.12,Na/Na+
